# WHO Gateway Database with Category Mapping from URLs

This notebook builds a database of WHO health indicators by:
1. Loading categories from `data_sets.json` with their proper codes (e.g., "EPW")
2. Storing category codes as strings in the database (e.g., "EPW: European Programme of Work")
3. Parsing indicator URLs to extract category codes (like "epw_38" → category "epw")
4. Linking each indicator to its proper category based on the URL pattern

## How It Works

For a URL like `https://gateway.euro.who.int/en/indicators/epw_38-ihr-average-of-capacities/#id=36879`:
1. The code extracts "epw" as the category code
2. It looks up this code in the database categories
3. It links the indicator to the "EPW: European Programme of Work" category

## Process Flow

1. Create database schema with category code field
2. Load categories from `data_sets.json`
3. Store both the category code and name in the database
4. Parse sitemap and extract indicators
5. Match indicators to categories based on URL patterns
6. View results and statistics with category codes

In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import re
import time
from urllib.parse import urljoin
import random
import os
import json


In [ ]:
# Create SQLite database
conn = sqlite3.connect('ind.db')
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE IF NOT EXISTS categories (
    id INTEGER PRIMARY KEY,
    code TEXT UNIQUE,
    name TEXT,
    url TEXT UNIQUE,
    parent_id INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS indicators (
    id INTEGER PRIMARY KEY,
    name TEXT,
    url TEXT UNIQUE,
    category_id INTEGER,
    description TEXT
)
''')

In [ ]:
# Function to download and save the sitemap HTML locally
def download_and_save_sitemap(url, save_path="sitemap.html"):
    """
    Downloads the sitemap from the given URL and saves it locally
    
    Args:
        url: URL of the sitemap
        save_path: Local file path to save the HTML
        
    Returns:
        True if successful, False otherwise
    """
    print(f"Attempting to download sitemap from: {url}")
    
    # Create browser-like headers to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://gateway.euro.who.int/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none'
    }
    
    try:
        # First try with complete headers
        response = requests.get(url, headers=headers, timeout=30)
        
        # If we still get 403, try with a different approach
        if response.status_code == 403:
            print("Got 403 with first attempt, trying with a simpler approach...")
            
            # Create a session to maintain cookies
            session = requests.Session()
            
            # First access the main site to get cookies
            main_url = "https://gateway.euro.who.int/en/"
            session.get(main_url, headers=headers)
            
            # Then try to access the sitemap
            response = session.get(url, headers=headers, timeout=30)
        
        # Check the response
        if response.status_code == 200:
            # Save the HTML content to a file
            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            print(f"Successfully downloaded and saved sitemap to: {save_path}")
            return True
        else:
            print(f"Failed to download sitemap. Status code: {response.status_code}")
            print(f"Response headers: {response.headers}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False



In [ ]:
# Try to download the sitemap Execution
sitemap_url = "https://gateway.euro.who.int/en/htmlsitemap/"
sitemap_file = "who_sitemap.html"
download_success = download_and_save_sitemap(sitemap_url, sitemap_file)

In [ ]:
# Function to create categories from data_sets.json
def create_categories_from_json(json_file_path="about/data_sets.json"):
    """
    Create categories from data_sets.json file
    
    Args:
        json_file_path: Path to the data_sets.json file
        
    Returns:
        Dictionary mapping dataset codes to category IDs
    """
    try:
        # Load the JSON file
        with open(json_file_path, 'r', encoding='utf-8') as f:
            datasets = json.load(f)
            
        print(f"Loaded {len(datasets)} datasets from {json_file_path}")
        
        # Connect to database
        conn = sqlite3.connect('ind.db')
        cursor = conn.cursor()
        
        # First, clear existing categories table
        cursor.execute("DELETE FROM categories")
        conn.commit()
        print("Cleared existing categories")
        
        # Dictionary to store mapping between dataset codes and category IDs
        code_to_id_map = {}
        
        # Add each dataset as a category
        for dataset in datasets:
            code = dataset['code']  # Keep original case for display
            code_lower = code.lower()  # Use lowercase for matching
            name = dataset['short_name']
            full_name = dataset['full_name']
            url = dataset['url']
            
            # Format the display name with code prefix
            display_name = f"{code}: {full_name}"
            
            # Insert into categories
            try:
                cursor.execute(
                    "INSERT INTO categories (code, name, url) VALUES (?, ?, ?)",
                    (code, display_name, url)
                )
                # Get the ID of the inserted category
                category_id = cursor.lastrowid
                code_to_id_map[code_lower] = category_id
                conn.commit()
                print(f"Added category: {display_name} (ID: {category_id})")
            except sqlite3.IntegrityError:
                print(f"Duplicate category: {code}")
            
        conn.close()
        print(f"Created {len(code_to_id_map)} categories from datasets")
        return code_to_id_map
        
    except FileNotFoundError:
        print(f"File not found: {json_file_path}")
        return {}
    except json.JSONDecodeError:
        print(f"Invalid JSON in file: {json_file_path}")
        return {}
    except Exception as e:
        print(f"Error creating categories: {str(e)}")
        return {}


In [ ]:
# Create categories and get code-to-id mapping
dataset_code_map = create_categories_from_json("about/data_sets.json")

In [ ]:
# Function to load sitemap from file and parse with BeautifulSoup
def load_sitemap_from_file(file_path="who_sitemap.html"):
    """
    Load a sitemap from a local file and parse it with BeautifulSoup
    
    Args:
        file_path: Path to the HTML file
        
    Returns:
        BeautifulSoup object or None if failed
    """
    if not os.path.exists(file_path):
        print(f"Sitemap file not found: {file_path}")
        return None
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        soup = BeautifulSoup(html_content, 'html.parser')
        print(f"Loaded sitemap from {file_path}")
        return soup
    except Exception as e:
        print(f"Error loading sitemap: {e}")
        return None

# Load the sitemap if file exists, otherwise use existing sitemap_soup if available
sitemap_file = "who_sitemap.html"
if os.path.exists(sitemap_file):
    sitemap_soup = load_sitemap_from_file(sitemap_file)
elif 'sitemap_soup' not in locals() or sitemap_soup is None:
    print("No sitemap available. Please download the sitemap first.")
    # You can uncomment the next line to download if needed
    # download_and_save_sitemap(sitemap_url, sitemap_file)

In [ ]:
#print(sitemap_soup)
print(dataset_code_map)


In [ ]:
# Function to extract indicators from sitemap and link to categories based on URL pattern
def extract_indicators_from_sitemap(sitemap_soup, dataset_code_map):
    """
    Extract indicators from sitemap and link them to categories based on URL pattern
    
    Args:
        sitemap_soup: BeautifulSoup object of the parsed sitemap
        dataset_code_map: Dictionary mapping dataset codes to category IDs
    """
    # Define the base URL for the WHO Gateway site
    base_url = "https://gateway.euro.who.int"
    
    if not sitemap_soup:
        print("No sitemap soup provided")
        return
    
    if not dataset_code_map:
        print("No dataset code map provided")
        return
        
    print(f"Starting extraction from sitemap with base URL: {base_url}")
    print(f"Dataset code map contains {len(dataset_code_map)} entries")
    
    # Connect to database
    conn = sqlite3.connect('ind.db')
    cursor = conn.cursor()
    
    # Clear existing indicators
    cursor.execute("DELETE FROM indicators")
    conn.commit()
    print("Cleared existing indicators")
    
    # Get all category codes from the database to use for matching
    cursor.execute("SELECT id, code FROM categories")
    db_category_codes = {row[1].lower(): row[0] for row in cursor.fetchall()}
    print(f"Loaded {len(db_category_codes)} category codes from database")
    
    # Extract all links from the sitemap
    links = sitemap_soup.find_all('a')
    print(f"Found {len(links)} links in the sitemap")
    
    # Regular expression to extract category codes from URLs
    # Pattern like: /indicators/DATASET_CODE_digits-name/
    url_pattern = re.compile(r'/indicators/([a-zA-Z_]+)[-_]?\d+.*?/')
    
    # Keep track of indicators processed
    indicators_added = 0
    matched_codes = set()
    unmatched_codes = set()
    
    # Process each link
    for link in links:
        href = link.get('href')
        if not href:
            continue
            
        url = urljoin(base_url, href)
        name = link.text.strip()
        
        # Skip empty links
        if not name or url == base_url:
            continue
            
        # Check if this looks like an indicator URL
        if '/indicators/' in url or '/indicator/' in url:
            # Try to extract dataset code from URL
            match = url_pattern.search(url)
            
            if match:
                # Extract the code from the URL
                extracted_code = match.group(1).lower()
                
                # Find the category ID for this code
                category_id = None
                
                # Try exact match first with database category codes
                if extracted_code in db_category_codes:
                    category_id = db_category_codes[extracted_code]
                    matched_codes.add(extracted_code)
                # Then try with our dataset code map
                elif extracted_code in dataset_code_map:
                    category_id = dataset_code_map[extracted_code]
                    matched_codes.add(extracted_code)
                else:
                    # Try to find a partial match
                    for code in db_category_codes:
                        if extracted_code.startswith(code.lower()) or code.lower().startswith(extracted_code):
                            category_id = db_category_codes[code.lower()]
                            matched_codes.add(code.lower())
                            break
                    
                    if category_id is None:
                        unmatched_codes.add(extracted_code)
                        
                # Insert the indicator with the category ID
                try:
                    cursor.execute(
                        "INSERT INTO indicators (name, url, category_id) VALUES (?, ?, ?)",
                        (name, url, category_id)
                    )
                    indicators_added += 1
                    if indicators_added % 100 == 0:
                        print(f"Added {indicators_added} indicators...")
                        conn.commit()
                except sqlite3.IntegrityError:
                    # Skip duplicates
                    pass
    
    # Final commit
    conn.commit()
    
    # Print results
    print(f"Extraction complete. Added {indicators_added} indicators.")
    print(f"Matched codes: {', '.join(sorted(matched_codes))}")
    print(f"Unmatched codes: {', '.join(sorted(unmatched_codes))}")
    
    conn.close()



In [ ]:
# Execute the extraction with the loaded sitemap
if 'sitemap_soup' in locals() and sitemap_soup is not None and 'dataset_code_map' in locals() and dataset_code_map:
    print("Running indicator extraction...")
    extract_indicators_from_sitemap(sitemap_soup, dataset_code_map)
else:
    print("Missing required variables. Make sure sitemap_soup and dataset_code_map are available.")

# URL-Based Category Mapping Process

This section extracts indicators from the WHO sitemap and links them to categories based on URL patterns:

1. **URL Pattern Extraction**: From a URL like `https://gateway.euro.who.int/en/indicators/epw_38-ihr-average-of-capacities/#id=36879`, we extract the category code `epw`

2. **Category Matching Process**:
   - First try direct matching with known category codes (e.g., `epw` → `EPW`)
   - Then try partial matching for variations (e.g., `epw_cp` might match with `epw`)
   - Store the mapped category ID with each indicator

3. **Results**: 
   - We track which codes were successfully matched and which remain unmatched
   - This helps identify any categories that might need to be added manually

In [ ]:
# Function to view the database contents and statistics
def view_database_stats():
    """
    View the database contents and statistics
    """
    conn = sqlite3.connect('ind.db')
    cursor = conn.cursor()
    
    # Get category count
    cursor.execute("SELECT COUNT(*) FROM categories")
    category_count = cursor.fetchone()[0]
    print(f"Categories: {category_count}")
    
    # Get indicator count
    cursor.execute("SELECT COUNT(*) FROM indicators")
    indicator_count = cursor.fetchone()[0]
    print(f"Indicators: {indicator_count}")
    
    # Get all categories with their codes
    cursor.execute("SELECT id, code, name FROM categories ORDER BY code")
    print("\nAll categories with codes:")
    for id, code, name in cursor.fetchall():
        print(f"- [{id}] {name}")
    
    # Get indicator count by category
    cursor.execute("""
        SELECT c.code, c.name, COUNT(i.id) as count
        FROM categories c
        LEFT JOIN indicators i ON c.id = i.category_id
        GROUP BY c.id
        ORDER BY count DESC
    """)
    
    print("\nIndicator count by category:")
    for code, name, count in cursor.fetchall():
        print(f"- {code}: {count} indicators")
    
    # Get indicators without categories
    cursor.execute("SELECT COUNT(*) FROM indicators WHERE category_id IS NULL")
    uncategorized = cursor.fetchone()[0]
    print(f"\nUncategorized indicators: {uncategorized}")
    
    # Sample of indicators with their categories
    cursor.execute("""
        SELECT i.name, i.url, c.code, c.name as category_name
        FROM indicators i
        LEFT JOIN categories c ON i.category_id = c.id
        ORDER BY RANDOM()
        LIMIT 5
    """)
    
    print("\nSample indicators:")
    for name, url, code, category in cursor.fetchall():
        category_info = f"{code}: {category}" if code else "Unknown"
        print(f"- {name}")
        print(f"  Category: {category_info}")
        print(f"  URL: {url}")
    
    conn.close()

# View the database statistics
view_database_stats()

In [ ]:
# Function to export categories and indicators to another database
def export_to_external_db(external_db_path="db1.sql"):
    """
    Export categories and indicators to another database file
    
    Args:
        external_db_path: Path to the external database file
    """
    print(f"Exporting data to external database: {external_db_path}")
    
    # Connect to our current database
    source_conn = sqlite3.connect('ind.db')
    source_cursor = source_conn.cursor()
    
    # Connect to external database
    try:
        target_conn = sqlite3.connect(external_db_path)
        target_cursor = target_conn.cursor()
        
        # Create categories table in target database if it doesn't exist
        target_cursor.execute('''
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY,
            code TEXT,
            name TEXT,
            url TEXT,
            parent_id INTEGER
        )
        ''')
        
        # Create indicators table in target database if it doesn't exist
        target_cursor.execute('''
        CREATE TABLE IF NOT EXISTS indicators (
            id INTEGER PRIMARY KEY,
            name TEXT,
            url TEXT,
            category_id INTEGER,
            description TEXT
        )
        ''')
        
        # Export categories
        source_cursor.execute("SELECT id, code, name, url, parent_id FROM categories")
        categories = source_cursor.fetchall()
        print(f"Exporting {len(categories)} categories...")
        
        # Clear existing categories in target
        target_cursor.execute("DELETE FROM categories")
        
        # Insert categories into target
        for category in categories:
            try:
                target_cursor.execute(
                    "INSERT INTO categories (id, code, name, url, parent_id) VALUES (?, ?, ?, ?, ?)",
                    category
                )
            except sqlite3.IntegrityError:
                print(f"Error inserting category ID {category[0]}")
        
        # Export indicators
        source_cursor.execute("SELECT id, name, url, category_id, description FROM indicators")
        indicators = source_cursor.fetchall()
        print(f"Exporting {len(indicators)} indicators...")
        
        # Clear existing indicators in target
        target_cursor.execute("DELETE FROM indicators")
        
        # Insert indicators into target
        for indicator in indicators:
            try:
                target_cursor.execute(
                    "INSERT INTO indicators (id, name, url, category_id, description) VALUES (?, ?, ?, ?, ?)",
                    indicator
                )
            except sqlite3.IntegrityError:
                print(f"Error inserting indicator ID {indicator[0]}")
        
        # Commit changes
        target_conn.commit()
        print("Export completed successfully")
        
        # Close connections
        target_conn.close()
        
    except Exception as e:
        print(f"Error exporting to external database: {e}")
    
    # Close source connection
    source_conn.close()

# Run the export if needed
# export_to_external_db("db1.sql")

Populate database ind.db with indicator code


In [4]:
import re

def add_indicator_code_column():
    """
    Add a new column 'indicator_code' to the indicators table and populate it
    by extracting the code from the URL pattern.
    
    URL Pattern: /indicators/CODE-.../
    Examples:
    - https://gateway.euro.who.int/en/indicators/hfamdb_125-deaths-asthma/ → hfamdb_125
    - https://gateway.euro.who.int/en/indicators/hfa_329-2091-number-of-new-malaria-cases/ → hfa_329
    """
    conn = sqlite3.connect('intro/ind.db')
    cursor = conn.cursor()
    
    try:
        # Check if the column already exists
        cursor.execute("PRAGMA table_info(indicators)")
        columns = [column[1] for column in cursor.fetchall()]
        
        if 'indicator_code' not in columns:
            # Add the new column
            cursor.execute("ALTER TABLE indicators ADD COLUMN indicator_code TEXT")
            print("Added 'indicator_code' column to indicators table")
        else:
            print("'indicator_code' column already exists")
        
        # Regular expression to extract indicator code from URL
        # Pattern: /indicators/CODE-.../ or /indicators/CODE_.../
        # Examples: /indicators/hfamdb_125-deaths-asthma/ → hfamdb_125
        #           /indicators/hfa_329-2091-number-of-new-malaria-cases/ → hfa_329
        url_pattern = re.compile(r'/indicators/([a-zA-Z_]+[a-zA-Z0-9_]*?)(?:[-_]|$)')
        
        # Get all indicators with their URLs
        cursor.execute("SELECT id, url FROM indicators")
        indicators = cursor.fetchall()
        
        print(f"Processing {len(indicators)} indicators...")
        
        updated_count = 0
        failed_count = 0
        
        for indicator_id, url in indicators:
            if not url:
                continue
                
            # Extract the indicator code from the URL
            match = url_pattern.search(url)
            
            if match:
                indicator_code = match.group(1)
                
                # Update the indicator with the extracted code
                cursor.execute(
                    "UPDATE indicators SET indicator_code = ? WHERE id = ?",
                    (indicator_code, indicator_id)
                )
                updated_count += 1
                
                if updated_count % 100 == 0:
                    print(f"Updated {updated_count} indicators...")
                    conn.commit()
            else:
                failed_count += 1
                print(f"Failed to extract code from URL: {url}")
        
        # Final commit
        conn.commit()
        
        print(f"\nUpdate complete!")
        print(f"Successfully updated: {updated_count} indicators")
        print(f"Failed to extract code: {failed_count} indicators")
        
        # Show some examples of the extracted codes
        cursor.execute("""
            SELECT name, url, indicator_code 
            FROM indicators 
            WHERE indicator_code IS NOT NULL 
            ORDER BY RANDOM() 
            LIMIT 5
        """)
        
        print("\nSample extracted indicator codes:")
        for name, url, code in cursor.fetchall():
            print(f"- {name}")
            print(f"  Code: {code}")
            print(f"  URL: {url}")
            print()
        
    except Exception as e:
        print(f"Error adding indicator_code column: {e}")
        conn.rollback()
    finally:
        conn.close()

# Execute the function to add the indicator_code column
add_indicator_code_column()


Added 'indicator_code' column to indicators table
Processing 2429 indicators...
Updated 100 indicators...
Updated 200 indicators...
Updated 300 indicators...
Updated 400 indicators...
Updated 500 indicators...
Updated 600 indicators...
Updated 700 indicators...
Updated 800 indicators...
Updated 900 indicators...
Updated 1000 indicators...
Updated 1100 indicators...
Updated 1200 indicators...
Updated 1300 indicators...
Updated 1400 indicators...
Updated 1500 indicators...
Updated 1600 indicators...
Updated 1700 indicators...
Updated 1800 indicators...
Updated 1900 indicators...
Updated 2000 indicators...
Updated 2100 indicators...
Updated 2200 indicators...
Updated 2300 indicators...
Updated 2400 indicators...

Update complete!
Successfully updated: 2429 indicators
Failed to extract code: 0 indicators

Sample extracted indicator codes:
- Youth unemployment rate, % of labor force ages 15-24, male
  Code: hfa_639
  URL: https://gateway.euro.who.int/en/indicators/hfa_639-youth-unemployment